# Level 4, Notebook 1: Tenant limits and cost control

**Problems in this notebook**

- Problem 31: Rate-Limit Model Calls per Tenant
- Problem 35: API Quota Enforcement
- Problem 39: Cost Anomaly Detection

**How to use this notebook**

1. Read the problem and say your plan aloud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `01_tenant_limits_and_cost_THEORY.md` in this folder.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 31: Rate-Limit Model Calls per Tenant

**The situation**

Your platform sends model calls for many customers (tenants) through one shared account. Last week one tenant ran a loop that used most of the capacity, and every other customer saw slow answers.

**Your task**

Build a fixed-window rate limiter. Each tenant may make at most `limit` calls per window of `window_seconds`. The current time is passed in as `now`, so tests never wait on a real clock.

**Example**

`limit=2, window_seconds=60`. Calls from `t1` at seconds 1, 2, 3 give `True, True, False`. A call from `t2` at second 3 is still `True`.

**Ask before coding**

- Is the limit per tenant, per user, or per model?
- Fixed window is fine here, or do you want a sliding window or token bucket?
- What should happen if a timestamp arrives that is older than the last one we saw?

**Plan to say aloud**

1. Turn the time into a window number: `now // window_seconds`.
2. Keep one record per tenant: the window number and how many calls it used.
3. If the window is newer than the stored one, reset the count to zero.
4. Allow and add one if the count is below the limit. Otherwise refuse.

### Prompt for your AI assistant

```text
Write a Python class TenantRateLimiter with __init__(self, limit: int, window_seconds: int) and allow(self, tenant_id: str, now: float) -> bool. It is a fixed-window limiter: window = int(now // window_seconds). Store one (window, count) pair per tenant so old windows do not pile up. Reset the count only when the window is newer than the stored one; an older timestamp counts against the current window. Return True and add one when count < limit, else return False. Do not call time.time(). Standard library only. Show only the class.
```

**Check the AI's code for**

- One global counter instead of a counter per tenant, so one tenant blocks everyone.
- Calls `time.time()` inside `allow` instead of using the `now` argument, which makes tests slow and flaky.
- Keeps a dict keyed by `(tenant, window)` that grows forever, with no cleanup of old windows.
- Resets the count when a timestamp goes backwards, which lets a caller dodge the limit.

In [ ]:
class TenantRateLimiter:
    """Fixed-window rate limiter, one counter per tenant.

    Rules:
    - window = int(now // window_seconds)
    - keep one (window, count) pair per tenant
    - a newer window resets the count; an older timestamp counts against the current window
    - allow (and add one) only while count < limit
    """

    def __init__(self, limit: int, window_seconds: int):
        self.limit = limit
        self.window_seconds = window_seconds

    def allow(self, tenant_id: str, now: float) -> bool:
        raise NotImplementedError

In [ ]:
def replay_limiter(limiter_class, limit, window_seconds, calls):
    """Feed (tenant_id, now) calls to a fresh limiter and collect the answers."""
    limiter = limiter_class(limit, window_seconds)
    return [limiter.allow(tenant, now) for tenant, now in calls]

cases = [
    ((2, 60, [("t1", 1), ("t1", 2), ("t1", 3)]), [True, True, False]),
    ((2, 60, [("t1", 1), ("t1", 2), ("t2", 3), ("t1", 4)]), [True, True, True, False]),
    ((2, 60, [("t1", 59), ("t1", 59), ("t1", 60), ("t1", 61), ("t1", 62)]), [True, True, True, True, False]),
    ((2, 60, [("t1", 60), ("t1", 61), ("t1", 5)]), [True, True, False]),  # clock went backwards
    ((0, 60, [("t1", 1)]), [False]),
]
run_checks(lambda *a: replay_limiter(TenantRateLimiter, *a), cases)

### Reference solution

Try first. Then compare.

In [ ]:
class TenantRateLimiterReference:
    def __init__(self, limit: int, window_seconds: int):
        self.limit = limit
        self.window_seconds = window_seconds
        self.state: dict[str, tuple[int, int]] = {}  # tenant -> (window, count)

    def allow(self, tenant_id: str, now: float) -> bool:
        window = int(now // self.window_seconds)
        stored_window, count = self.state.get(tenant_id, (window, 0))
        if window > stored_window:
            stored_window, count = window, 0
        if count >= self.limit:
            return False
        self.state[tenant_id] = (stored_window, count + 1)
        return True

assert run_checks(lambda *a: replay_limiter(TenantRateLimiterReference, *a), cases)

**Explain it in 30 seconds**

> "I turn the time into a window number and keep one small record per tenant: which window it is in and how many calls it used. A newer window resets the count. If the count is already at the limit I say no. Because each tenant has its own record, one busy customer cannot use up another customer's capacity."

**Edge cases to mention**

- Burst at a window edge: 2 calls at second 59 and 2 more at second 60 all pass.
- A timestamp older than the last one seen (clock skew between servers).
- A limit of zero, for a suspended tenant.
- Different tenants on different plans need different limits.

**If they ask "how would this work in production?"**

- Several app servers each hold their own memory, so the count must live in a shared store such as Redis, or in the API gateway.
- Return a clear "try again in N seconds" answer (HTTP 429 with Retry-After) instead of a generic error.
- Use a sliding window or token bucket if edge bursts matter, and read limits from each tenant's plan.

## Problem 35: API Quota Enforcement

**The situation**

Each customer buys a monthly allowance of model calls or tokens. Before a new call goes out, the platform must check that the customer still has room in their allowance.

**Your task**

Write `can_make_model_call(tenant_id, usage, quota, requested=1)`. Return `True` only if the tenant has a quota and `usage + requested` stays within it. A tenant with no quota entry is refused.

**Example**

`usage={"t1": 99}, quota={"t1": 100}` gives `True`. `usage={"t1": 100}` gives `False`. A tenant missing from `quota` gives `False`.

**Ask before coding**

- Is the quota counted in calls, tokens, or money?
- If a tenant has no quota entry, should we refuse or allow?
- Do you want a soft warning before the hard stop?

**Plan to say aloud**

1. Look up the tenant's quota. If there is none, refuse (fail closed).
2. Look up current usage. A tenant with no usage yet starts at zero.
3. Allow only if usage plus the requested amount is within the quota.

### Prompt for your AI assistant

```text
Write a Python function can_make_model_call(tenant_id: str, usage: dict[str, int], quota: dict[str, int], requested: int = 1) -> bool. Rules: if tenant_id is not in quota, return False (never treat a missing quota as unlimited). Missing usage counts as 0. Return True only when usage + requested <= quota. Look up only this tenant's entries. Show only the function.
```

**Check the AI's code for**

- Treats a missing quota as unlimited (for example `quota.get(tenant_id, float('inf'))`).
- Uses `<` against the quota without adding `requested`, so a large token request slips past the limit.
- Raises `KeyError` for a new tenant who has a quota but no usage yet.
- Sums usage across all tenants instead of looking at this one tenant.

In [ ]:
def can_make_model_call(tenant_id: str, usage: dict[str, int], quota: dict[str, int], requested: int = 1) -> bool:
    """Return True if this tenant can spend `requested` more units.

    Rules:
    - tenant missing from quota -> False (fail closed, never unlimited)
    - tenant missing from usage -> usage is 0
    - allowed only when usage + requested <= quota
    """
    raise NotImplementedError

In [ ]:
cases = [
    (("t1", {"t1": 99}, {"t1": 100}), True),
    (("t1", {"t1": 100}, {"t1": 100}), False),
    (("t9", {}, {"t1": 100}), False),                      # no quota entry: refuse
    (("t2", {}, {"t2": 50}), True),                        # new tenant, no usage yet
    (("t1", {"t1": 60}, {"t1": 100}, 50), False),          # big token request would overshoot
    (("t1", {"t1": 0, "t2": 999}, {"t1": 10, "t2": 10}), True),  # other tenant's usage does not matter
]
run_checks(can_make_model_call, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def can_make_model_call_reference(tenant_id: str, usage: dict[str, int], quota: dict[str, int], requested: int = 1) -> bool:
    limit = quota.get(tenant_id)
    if limit is None:
        return False
    return usage.get(tenant_id, 0) + requested <= limit

assert run_checks(can_make_model_call_reference, cases)

**Explain it in 30 seconds**

> "I look up this tenant's quota first. If there is no quota I refuse, because guessing 'unlimited' can run up a surprise bill. Then I add the requested amount to current usage and allow the call only if it fits. Usage for a brand new tenant starts at zero."

**Edge cases to mention**

- Tenant missing from the quota table, often a setup mistake.
- A request bigger than the remaining allowance (a long prompt in tokens).
- A real unlimited plan, which should be an explicit flag, not a missing row.
- Two calls checked at the same moment, both see room, both go through.

**If they ask "how would this work in production?"**

- Check and add usage in one atomic step (a Redis increment or a database transaction) so parallel calls cannot overshoot.
- Track quota per billing period, with a soft warning (say at 80%) before the hard stop.
- Show customers a usage page and send alerts before they hit the limit.

## Problem 39: Cost Anomaly Detection

**The situation**

An agent bug can call the model in a loop and spend a month of budget in a day. The platform team wants a daily job that flags tenants whose spend jumped far above their normal level.

**Your task**

Write `detect_cost_anomalies(history, today, factor, min_spend=1.0)`. Flag a tenant when today's cost is more than `factor` times their average past cost and also at least `min_spend`. Return the flagged tenant ids sorted.

**Example**

`history={"t1": [10, 12, 11]}, today={"t1": 100}, factor=3` gives `["t1"]` (average 11, threshold 33, today 100).

**Ask before coding**

- How many past days count as the baseline?
- Is there a minimum spend before we bother alerting?
- Should a flag only alert, or also block the tenant?

**Plan to say aloud**

1. Loop over tenants that have history. Skip empty history: no baseline yet.
2. Average the past costs.
3. Read today's cost (missing means 0).
4. Flag if today is above factor times the average and at least the minimum spend.
5. Return the flagged ids sorted so the output is stable.

### Prompt for your AI assistant

```text
Write a Python function detect_cost_anomalies(history: dict[str, list[float]], today: dict[str, float], factor: float, min_spend: float = 1.0) -> list[str]. For each tenant with a non-empty history, compute the average. Flag the tenant if today.get(tenant, 0) > factor * average and today's cost >= min_spend. Skip tenants with empty history. Do not divide by zero. Return the flagged tenant ids sorted. Show only the function.
```

**Check the AI's code for**

- Divides by `len(costs)` without skipping empty history, so a new tenant crashes the job.
- Has no minimum spend, so a jump from 1 cent to 5 cents raises an alert.
- Skips tenants with a zero average, which hides a tenant going from 0 to 500 overnight.
- Returns tenants in dict order instead of sorted, which makes tests flaky.

In [ ]:
def detect_cost_anomalies(
    history: dict[str, list[float]],
    today: dict[str, float],
    factor: float,
    min_spend: float = 1.0,
) -> list[str]:
    """Return sorted tenant ids whose spend today looks abnormal.

    Rules:
    - skip tenants with empty history (no baseline yet)
    - missing today's cost counts as 0
    - flag when today > factor * average AND today >= min_spend
    """
    raise NotImplementedError

In [ ]:
cases = [
    (({"t1": [10, 12, 11]}, {"t1": 100}, 3), ["t1"]),
    (({"t1": [10, 10]}, {"t1": 25}, 3), []),                              # busy day, not a spike
    (({"t2": [0.01, 0.02]}, {"t2": 0.5}, 3, 5.0), []),                    # tiny baseline, tiny spend
    (({"t3": [0, 0]}, {"t3": 50}, 3), ["t3"]),                            # zero baseline, real spend
    (({"new": [], "t1": [5]}, {"new": 900}, 3), []),                      # no baseline yet, t1 missing today
    (({"b": [1, 1], "a": [2, 2]}, {"a": 10, "b": 10}, 3), ["a", "b"]),    # sorted output
]
run_checks(detect_cost_anomalies, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def detect_cost_anomalies_reference(
    history: dict[str, list[float]],
    today: dict[str, float],
    factor: float,
    min_spend: float = 1.0,
) -> list[str]:
    flagged = []
    for tenant, costs in history.items():
        if not costs:
            continue
        average = sum(costs) / len(costs)
        spent = today.get(tenant, 0.0)
        if spent > factor * average and spent >= min_spend:
            flagged.append(tenant)
    return sorted(flagged)

assert run_checks(detect_cost_anomalies_reference, cases)

**Explain it in 30 seconds**

> "For each tenant I average their past daily cost and compare today against a multiple of that average. I also require a minimum spend so tiny accounts do not trigger noise. Tenants with no history are skipped because there is nothing to compare against. I return the list sorted so the result is the same every run."

**Edge cases to mention**

- A new tenant with no history (watch it with a fixed budget alert instead).
- A zero average: any real spend is a spike, so do not skip it.
- A planned launch day that is a legitimate spike.
- Very small baselines where a factor of 3 is only a few cents.

**If they ask "how would this work in production?"**

- Alert a human first. Only block automatically if the customer's policy says so.
- Break cost down by model, endpoint, and user so the alert says what caused the spike.
- Run on a rolling window (hourly as well as daily) so a runaway loop is caught in minutes.

## Wrap-up

- Every check is per tenant. One customer's traffic or spend must never use up another customer's share.
- When data is missing (no quota, broken timestamp), choose the safe answer: refuse or skip, never "unlimited".
- Time and history are passed in as arguments, so the logic is easy to test and the real store (Redis, a database) can be swapped in later.